In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run ../1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"s3://sportsbar-dp-umesh/{data_source}"
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"
print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)

#define the tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

In [0]:
df = (spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(f"{landing_path}/*.csv")
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)

print("Total_Rows: ", df.count())
df.show(5)

In [0]:
display(df.limit(20))

In [0]:
df.write\
 .format("delta")\
 .option("delta.enableChangeDataFeed", "true")\
 .mode("append")\
 .saveAsTable(bronze_table)

In [0]:
files = dbutils.fs.ls(landing_path)

for file_info in files:
    dbutils.fs.mv(
        file_info.path, f"{processed_path}/{file_info.name}", True
    )

In [0]:
df_orders = spark.sql(f"select * from {bronze_table}")
df_orders.show(2)

In [0]:
# 1. Keep only rows where order_qty is present.
df_orders = df_orders.filter(F.col("order_qty").isNotNull())

# 2. Clean customer_id, keep numeric else set to 999999
df_orders = df_orders.withColumn("customer_id",
                    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
                     .otherwise("999999")
                     .cast("string")                                 
)

# 3. Remove weekday name from the date text
# "Tuesday, July 01, 2025" -> "July 01, 2025"
df_orders = df_orders.withColumn("order_placement_date",
                    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)